# Lecture 2: makemore — 字符级 Bigram 语言模型

本笔记本跟随 Andrej Karpathy 的 [Neural Networks: Zero to Hero](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) 系列课程第 2 讲，构建一个字符级的 bigram 语言模型来生成人名。

**学习路线：**
1. 数据加载与 bigram 统计（字典方法）
2. 用张量构建 27×27 计数矩阵并可视化
3. 将计数转为概率矩阵，采样生成名字
4. 用对数似然（log-likelihood）评估模型质量
5. 用神经网络（单层 Softmax）替代查表法
6. 梯度下降优化神经网络权重

## 1. 数据加载与探索

**概念解释：** 我们使用一个包含 32,000+ 人名的数据集 `names.txt`。每个名字是一个字符序列，我们的目标是学习字符之间的统计规律，从而生成新的、听起来合理的名字。

这是一个**生成式模型（Generative Model）**的入门案例：不是分类或预测，而是"创造"新数据。

In [1]:
# .read()       → 把整个文件读成一个字符串（包含换行符 \n）
#                  "emma\nolivia\nava\n..."
# .splitlines() → 按换行符 \n 拆成列表
#                  ["emma", "olivia", "ava", ...]
#
# 所以 word 是一个 list[str]，每个元素是一个名字
# word[0] = 'emma'（第一个名字），len(word) = 名字总数
word=open('D:\\Vault-4\\Projects\\makemore\\names.txt','r').read().splitlines()
word[:9], type(word), len(word)  # 显示前9个名字，数据类型和总名字数量

(['emma',
  'olivia',
  'ava',
  'isabella',
  'sophia',
  'charlotte',
  'mia',
  'amelia',
  'harper'],
 list,
 32033)

In [ ]:
# 对比：不加 .splitlines()，.read() 返回的是一整个字符串
# word 此时的类型是 str，不是 list
# word[:9] 取的是前 9 个【字符】（包括换行符 \n 也算一个字符）
# 例如：'emma\noliv'（e-m-m-a-\n-o-l-i-v = 9 个字符）
#
# 注意：str 和 list 都没有 .dtype 属性
# .dtype 是 torch.Tensor / numpy.ndarray 专有的
# 对普通 Python 对象，用 type(word) 查看类型
word=open('D:\\Vault-4\\Projects\\makemore\\names.txt','r').read()
word[:9], type(word), len(word)  # 显示前9个名字，数据类型和总名字数量

In [2]:
len_w=[len(w) for w in word] # len(w) 是每个名字的长度，len_w 是一个 list[int]，包含了所有名字的长度

In [3]:
len_w[:10], type(len_w), len(len_w), min(len_w), max(len_w), sum(len_w)/len(len_w)    # 显示前10个名字长度，数据类型和总名字数量  # 显示名字长度的最小值、最大值和平均值

([4, 6, 3, 8, 6, 9, 3, 6, 6, 6], list, 32033, 2, 15, 6.122217712983486)

## 2. Bigram 统计（字典方法）

**概念解释：** **Bigram** 是相邻的两个字符组成的对（pair）。例如 `emma` 产生的 bigram 有：`<S>e`, `em`, `mm`, `ma`, `a<E>`。

通过统计整个数据集中每个 bigram 出现的次数，我们就能知道"在某个字符后面，哪些字符更常出现"——这就是语言模型的本质。

**生活类比：** 想象你经常打字，输入法会提示"下一个字"——它就是基于类似的 bigram/n-gram 统计。

In [4]:
# b 是一个字典，用来统计每个 bigram（相邻字符对）出现的次数
# 键: (ch1, ch2) 元组，值: 出现次数
b={}

# 遍历前 2 个单词（word[:2] → ['emma', 'olivia']）
for w in word:

    # 构造带有【起始标记】和【终止标记】的字符列表
    # 例如 w = 'emma':
    #   list(w)  = ['e', 'm', 'm', 'a']
    #   chs      = ['<S>', 'e', 'm', 'm', 'a', '<E>']
    #
    # '<S>' 表示单词的开始（Start），'<E>' 表示单词的结束（End）
    # 这样我们就能学到：
    #   - '<S>' → 'e'：单词倾向以 'e' 开头
    #   - 'a' → '<E>'：单词倾向以 'a' 结尾
    chs=['<S>']+list(w)+['<E>'] 

    # 用 zip 滑动窗口提取所有相邻字符对（bigram）
    #
    # zip(chs, chs[1:]) 的原理：把列表和【错位一格的自己】逐位配对
    #   chs     = ['<S>', 'e', 'm', 'm', 'a', '<E>']
    #   chs[1:] = ['e',   'm', 'm', 'a', '<E>']
    #   zip 配对（以短的为准，'<E>' 没有对手被丢弃）：
    #     ('<S>','e'), ('e','m'), ('m','m'), ('m','a'), ('a','<E>')
    #
    # 等价于：
    #   for i in range(len(chs) - 1):
    #       ch1 = chs[i]
    #       ch2 = chs[i + 1]
    # 但 zip 写法更简洁、更 Pythonic
    for ch1, ch2 in zip(chs, chs[1:]):
        # bigram 用 tuple（元组）而不是 list（列表），原因：
        #   - tuple 不可变 → 可以当 dict 的 key（可哈希）
        #   - list  可变   → 不能当 dict 的 key（会报 TypeError）
        #
        # 对比：
        #   b[('e','m')] = 1   ✓  tuple 可以当 key
        #   b[['e','m']] = 1   ✗  list 不能当 key
        #
        # 简单记：固定数据/当 key → tuple()，需要增删改 → list[]
        bigram=(ch1, ch2)

        # dict.get(key, default) 的用法：
        #   d[key]             → 找不到就报 KeyError（激进派）
        #   d.get(key, default)→ 找不到就返回 default（温和派）
        #   d.get(key)         → 找不到就返回 None
        #
        # 举例：basket = {'apple': 3, 'banana': 5}
        #   basket.get('apple', 0)  → 3   （存在，返回实际值）
        #   basket.get('orange', 0) → 0   （不存在，返回默认值 0）
        #
        # 本行的执行过程（以 bigram ('m','m') 为例）：
        #   第一次遇到: b.get(('m','m'), 0) → 0（不存在）, b[('m','m')] = 0+1 = 1
        #   第二次遇到: b.get(('m','m'), 0) → 1（已存在）, b[('m','m')] = 1+1 = 2
        #
        # 等价于:
        #   if bigram not in b:
        #       b[bigram] = 0
        #   b[bigram] = b[bigram] + 1
        b[bigram]=b.get(bigram, 0)+1

In [ ]:
# 按出现次数从高到低排序，显示最常见的 10 个 bigram
# lambda x: x[1] 表示按字典 value（即计数）排序
# reverse=True 从大到小排列
sorted(b.items(), key=lambda x: x[1], reverse=True)[:10]

## 3. 计数矩阵（张量方法）

**概念解释：** 用 Python 字典统计 bigram 虽然直观，但效率不高。我们改用一个 **27×27 的二维张量 `N`** 来存储所有 bigram 的计数。

- 行 = 第一个字符（共 27 种：`.` + `a-z`）
- 列 = 第二个字符
- `N[i][j]` = 字符 `itos[i]` 后面紧跟字符 `itos[j]` 的次数

$$N_{ij} = \text{count}(\text{char}_i \to \text{char}_j)$$

**生活类比：** 这就像一张"交叉频率表"——类似于调查"喜欢咖啡的人中有多少也喜欢茶"的统计表格。

In [6]:
import torch

In [7]:
torch.zeros((27,27), dtype=torch.int32).dtype

torch.int32

#### 📐 PyTorch 数据类型（dtype）速查

浮点数在内存中按**科学计数法**存储，拆为三部分：`± 尾数 × 2^指数`。位数越多，精度和范围越大，但也越占内存。

| dtype | 位数 | 每个数占内存 | 说明 |
|---|---|---|---|
| `torch.float16` / `half` | 16-bit | 2 bytes | 指数 5 位，尾数 10 位；省内存但易溢出 |
| `torch.bfloat16` | 16-bit | 2 bytes | 指数 **8** 位，尾数 **7** 位；Google Brain 发明，范围≈float32，训练更稳 |
| **`torch.float32`** / `float` | 32-bit | 4 bytes | **默认类型**，日常训练首选 |
| `torch.float64` / `double` | 64-bit | 8 bytes | 超高精度，DL 中很少使用 |
| `torch.int32` / `int` | 32-bit | 4 bytes | 整数，适合**计数**（如下方的 N 矩阵） |
| `torch.int64` / `long` | 64-bit | 8 bytes | `CrossEntropyLoss` 要求标签为 long |
| `torch.bool` | 8-bit | 1 byte | True/False 掩码 |

> **类型提升（type promotion）：** 整数张量参与除法时自动升为浮点，例如 `N.float() / N.sum()` → float32。

#### 💡 Bit、Byte 与内存基础

**Bit（位）** 是最小数据单位，只能存 `0` 或 `1`（一个开关）。**Byte（字节）= 8 bits**，是 CPU 从内存读写的最小单位。

```
1 byte  = 8 bits
1 KB    = 1024 bytes
1 MB    = 1024 KB
1 GB    = 1024 MB
```

所以 dtype 的位数 ÷ 8 = 每个数占多少 bytes：float32 → 4 bytes，float16 → 2 bytes。

**为什么 1 byte = 8 bits？** IBM System/360（1964）确立了这个标准：
- 8 bit 能表示 256 种值，刚好够一个字符（英文字母 + 数字 + 符号）
- 8 = 2³，是 2 的幂，硬件电路天然按 2 的幂扩展，地址计算只需移位，不需除法
- 两位十六进制（0x00~0xFF）刚好对应 1 byte，方便人类读写

**为什么不用可变长 bit？** 固定长度才能**随机访问**——读第 500 个数只需 `500 × 8` 直接跳过去。可变长度必须从头数到第 499 个才知道第 500 个在哪，CPU 每秒几十亿次访问，扫描式寻址不可接受。不过在不需要随机访问、追求压缩率的场景（UTF-8、zip、H.264 视频编码）中确实使用可变长编码。

#### 🔍 int32 vs float32：同样 32 bits，分配方式完全不同

```
int32（32 bits）：
┌──────────────────────────────────┐
│ 1 bit 符号 │   31 bits 直接存整数值   │  → 没有指数、没有小数，精确整数
└──────────────────────────────────┘

float32（32 bits）：
┌──────────────────────────────────────────┐
│ 1 bit 符号 │ 8 bits 指数 │ 23 bits 尾数 │  → 科学计数法 ± 尾数 × 2^指数
└──────────────────────────────────────────┘
```

|  | int32 | float32 |
|---|---|---|
| 有指数吗 | **没有** | 有（8 bit） |
| 能存小数 | 不能 | 能 |
| 精确吗 | **精确** | 有浮点误差 |
| 范围 | ±21 亿 | ±3.4×10³⁸ |

这就是为什么下方计数矩阵 `N` 用 `int32`（计数是精确整数），而算概率时要 `N.float()` 转为 `float32`（概率是小数）。

**浮点误差：** 有些十进制小数在二进制中无限循环（就像十进制下 1/3 = 0.333...），float 只能截断近似。例如 `0.1 + 0.2 = 0.30000000000000004`。对神经网络训练的影响：学习率过大时参数变为大数，浮点精度不够，梯度更新会被"吞掉"——这是学习率需要合适的底层原因之一。

#### 📦 延伸：LLM 量化（Quantization）

量化 = 训练完成后，把模型参数从高精度（float32/float16）压缩为低精度（int8/int4），让模型更小更快。

```
float16 → int8 → int4：精度逐步降低，但模型体积减半再减半
70B 参数模型：float16 = 140 GB → int4 = 35 GB（一张 4090 就能跑）
```

**原理：** 将浮点数等比映射到整数区间。例如权重范围 [-1.0, 1.0] 映射到 int8 的 [-128, 127]，推理时再近似还原。神经网络有很强的冗余性，轻微的精度损失对最终输出影响很小。

**常见方案：** INT8（服务端部署）、INT4/GPTQ/AWQ（本地跑大模型）、GGUF Q4（llama.cpp）。Ollama 跑模型默认就是 Q4 量化。

> 这是 dtype 选择在工程上的极致延伸：训练用 float32 保精度 → 部署时逐步降精度换速度和内存。

In [8]:
# N=torch.zeros((28,28), dtype=torch.int32)
N=torch.zeros((27,27), dtype=torch.int32)  # 27×27 计数矩阵：26 个字母 + 1 个特殊符号 '.'

In [ ]:
word[:10]  # 重新确认 word 是列表形式（之前的 cell 覆盖过 word 变量）

In [ ]:
# 把所有名字拼成一个长字符串（无分隔符）
# 目的：提取所有出现过的字符，用于构建字符表
''.join(word)

In [ ]:
# set() 去重：提取所有不同字符
# 结果是 26 个英文字母（名字中只有小写字母，没有 '.'）
set(''.join(word))

### 构建字符-索引映射

接下来把字符转为整数索引。张量只能存数字，不能存字符——所以我们需要一个"翻译表"：`stoi`（字符→索引）和 `itos`（索引→字符）。特殊符号 `'.'` 同时充当起始符和终止符（代替之前的 `<S>` 和 `<E>`），分配索引 0。

In [ ]:
# 转为列表以便查看——注意 set 是无序的，每次运行顺序可能不同
list(set(''.join(word)))

In [ ]:
# sorted() 让字符按字母表顺序排列：['a', 'b', ..., 'z']
# 这 26 个字母将映射到索引 1-26，索引 0 留给特殊符号 '.'
chars = sorted(list(set(''.join(word))))
chars

In [ ]:
# ============================================================
# 构建 字符→索引 映射（stoi = "string to integer"）
# enumerate(chars) 从 0 开始编号，但我们 +1 让字母从 1 开始
# 特殊符号 '.' 手动指定为 0（同时充当起始符和终止符）
# ============================================================
stoi = {s: i+1 for i, s in enumerate(chars)}  # a→1, b→2, ..., z→26
stoi['.'] = 0                                  # '.' → 0（起止符）
stoi, type(stoi)

In [ ]:
# 反向映射：索引→字符（itos = "integer to string"）
# 生成名字时需要把索引转回可读字符
itos = {i: s for s, i in stoi.items()}         # 0→'.', 1→'a', ..., 26→'z'
itos

In [42]:
for w in word:                                    # 遍历所有名字
    chs=['.']+list(w)+['.']                       # 添加起止符号 '.'
    for ch1, ch2 in zip(chs, chs[1:]):            # 取相邻字符对（bigram）
        ix1=stoi[ch1]                             # 前一个字符的索引
        ix2=stoi[ch2]                             # 后一个字符的索引
        N[ix1, ix2]+=1                            # 在计数矩阵对应位置 +1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# %matplotlib inline 让图表直接显示在 notebook 中，而不是弹出新窗口

In [ ]:
# ============================================================
# 可视化 27×27 计数矩阵（热力图 + 文字标注）
# 颜色越深 = bigram 出现次数越多
# 每个格子同时显示：字符对（如 "ab"）和计数值
# ============================================================
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')                       # 用蓝色热力图显示计数矩阵
for i in range(N.shape[0]):                       # 遍历每一行（第一个字符）
    for j in range(N.shape[1]):                   # 遍历每一列（第二个字符）
        chstr = itos[i] + itos[j]                 # 该位置对应的 bigram 字符对
        plt.text(j, i, chstr, ha='center', va='bottom', color='gray')
        plt.text(j, i, N[i,j].item(), ha='center', va='top', color='gray')  # 显示计数值
    plt.axis('off')

## 4. 概率矩阵与采样

**概念解释：** 计数矩阵告诉我们"发生了多少次"，但要生成新名字，我们需要的是**概率**——"下一个字符是什么的可能性"。

将计数矩阵的每一行归一化（除以该行之和），就得到了概率矩阵 $P$：

$$P_{ij} = \frac{N_{ij}}{\sum_j N_{ij}}$$

每一行 $P_i$ 是一个概率分布（和为 1），表示"在字符 $i$ 之后，各字符出现的概率"。

然后用 `torch.multinomial` 按概率抽样，就能逐字符生成名字。

**生活类比：** 像扔一个不均匀的骰子——每面的概率不同，由训练数据决定。

In [ ]:
# N[0,:] 和 N[0] 等价：取第 0 行（'.' 后面跟各字符的计数）
# N[0, 0] = 0 表示 '.' 后面从不跟 '.'（不会连续出现两个边界符）
# N[0, 1] = 4410 表示有 4410 个名字以 'a' 开头
N[0,:]
N[0]

In [ ]:
# 演示单行归一化：把第 0 行的计数转为概率
p = N[0].float()       # 转为浮点数（int 不能做除法得到小数）
p = p / p.sum()        # 归一化：每个元素 / 总和 → 概率分布（和为 1）
p                      # p[1]=0.1377 意味着 13.77% 的名字以 'a' 开头

> **深入理解：`keepdim=True` 与广播机制**
>
> 当我们对整个矩阵做 `P.sum(1, keepdim=True)` 时，`keepdim=True` 至关重要。
> 没有它，`P.sum(1)` 返回形状 `(27,)`（一维向量），而 `P` 是 `(27,27)`。
> PyTorch 的广播规则会把 `(27,)` 当作"列向量"广播——每一**列**除以同一个数，
> 这完全搞反了！加上 `keepdim=True` 后结果形状是 `(27,1)`，广播时每一**行**
> 除以自己的行和，才是正确的按行归一化。
>
> **类比：** 想象一张成绩表，每行是一个学生的各科分数。你想把每个学生的分数
> 转为百分比（除以该学生的总分）。如果不小心除以了"每科的总分"，那就是按列
> 归一化——变成了"在这科中，这个学生占多少"，完全变了含义。

In [47]:
P=N.float()
P=P/P.sum(1, keepdim=True)  # 按行归一化：每行除以该行之和；keepdim=True 保持维度以便广播
P

tensor([[0.0000e+00, 1.3767e-01, 4.0770e-02, 4.8138e-02, 5.2758e-02, 4.7794e-02,
         1.3018e-02, 2.0885e-02, 2.7284e-02, 1.8450e-02, 7.5610e-02, 9.2498e-02,
         4.9074e-02, 7.9231e-02, 3.5776e-02, 1.2300e-02, 1.6077e-02, 2.8720e-03,
         5.1166e-02, 6.4153e-02, 4.0833e-02, 2.4350e-03, 1.1738e-02, 9.5839e-03,
         4.1832e-03, 1.6702e-02, 2.9001e-02],
        [1.9596e-01, 1.6408e-02, 1.5966e-02, 1.3870e-02, 3.0751e-02, 2.0422e-02,
         3.9546e-03, 4.9579e-03, 6.8821e-02, 4.8694e-02, 5.1645e-03, 1.6763e-02,
         7.4605e-02, 4.8222e-02, 1.6048e-01, 1.8592e-03, 2.4199e-03, 1.7707e-03,
         9.6326e-02, 3.2994e-02, 2.0274e-02, 1.1244e-02, 2.4613e-02, 4.7514e-03,
         5.3711e-03, 6.0499e-02, 1.2838e-02],
        [4.3100e-02, 1.2136e-01, 1.4367e-02, 3.7807e-04, 2.4575e-02, 2.4764e-01,
         0.0000e+00, 0.0000e+00, 1.5501e-02, 8.2042e-02, 3.7807e-04, 0.0000e+00,
         3.8941e-02, 0.0000e+00, 1.5123e-03, 3.9698e-02, 0.0000e+00, 0.0000e+00,
         3.1834e-

In [ ]:
# ============================================================
# 一次性对整个矩阵做按行归一化，得到 27×27 概率矩阵 P
# P.sum(1, keepdim=True) 的 keepdim=True 让结果保持 (27,1) 形状
# 这样 (27,27) / (27,1) 才能正确广播（每行除以自己的行和）
# ============================================================
P = N.float()
P = P / P.sum(1, keepdim=True)   # 按行归一化：每行是一个概率分布
P

### 从概率矩阵采样生成名字

有了概率矩阵 $P$，生成名字的算法很简单：从 `'.'`（索引 0）出发，查 $P$ 的第 0 行得到"下一个字符"的概率分布，用 `torch.multinomial` 按概率抽样一个字符，然后用这个新字符查下一行，循环直到抽到 `'.'`（终止）。

`torch.multinomial(p, num_samples=1)` 的作用：给它一个概率向量 `p`，它按概率随机返回一个索引——概率越高的索引被选中的可能性越大，就像抛一个"加权骰子"。

In [ ]:
# ============================================================
# 用概率矩阵 P 采样生成 5 个名字
# 流程：从 '.'（ix=0）出发 → 查 P[ix] 得到下一个字符的概率分布
#       → multinomial 按概率抽样 → 重复直到抽到 '.'（终止）
# ============================================================
g = torch.Generator().manual_seed(2147483647)

for i in range(5):                                # 生成 5 个名字
    out = []
    ix = 0                                        # 从 '.'（起始符）开始
    while True:
        p = P[ix]                                 # 取当前字符对应的概率行

        # p=N[ix].float()
        # p=p/p.sum()

        # p=torch.ones(27)/27

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率采样下一个字符

        # ix=torch.multinomial(p, num_samples=1, replacement=True).item()  # 按概率采样下一个字符

        out.append(itos[ix])
        # print(itos[ix], end='')
        if ix == 0:                               # 采到 '.'（终止符）则结束
            break

    print(''.join(out))

In [ ]:
# [可选实验] 验证 multinomial 采样是否符合理论概率
# 抽取 100 次，统计每个字符被抽中的频率，与理论概率 p 对比
# 取消注释可以运行，观察实际频率是否接近 p 中的值
#
# from collections import Counter
#
# samples = [torch.multinomial(p, num_samples=1, replacement=True).item() for _ in range(100)]
# counts = Counter(samples)
#
# # 按频率从高到低排序，显示：字符 | 抽中次数 | 实际频率 | 理论概率
# print(f"{'字符':>4} {'次数':>4} {'实际频率':>8} {'理论概率':>8}")
# print("-" * 30)
# for ix, cnt in counts.most_common():
#     print(f"{itos[ix]:>4} {cnt:>4} {cnt/100:>8.2%} {p[ix].item():>8.2%}")

### 穿插练习：PyTorch 基础操作

以下几个 cell 是关于 `torch.multinomial`、张量创建与索引的基础练习，帮助熟悉 PyTorch 的核心 API。可跳过。

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)   # 随机生成 3 个正数（均匀分布 [0,1)）
p = p / p.sum()                  # 归一化为概率分布：三个数之和 = 1
p
# sum(p)

In [ ]:
# 按概率 p 抽样 20 次：索引 0 (概率60%) 出现最多，索引 2 (概率9%) 出现最少
torch.multinomial(p, num_samples=20, replacement=True, generator=g)

In [ ]:
# 创建全零整数张量，练习张量的创建和索引
a = torch.zeros((3,5), dtype=torch.int32)  # 3行5列的零矩阵
a

In [ ]:
a.dtype  # 查看数据类型：torch.int32（和创建时指定的一致）

In [ ]:
a[1,3] += 1  # 就地修改：第 1 行第 3 列 +1（模拟计数矩阵的累加操作）

In [ ]:
a  # 查看修改后的张量：第 1 行第 3 列变成了 1

In [ ]:
a[0,0] = 5  # 直接赋值（不是累加）：第 0 行第 0 列设为 5

In [ ]:
a  # 确认赋值结果：[0,0]=5, [1,3]=1，其余为 0

## 5. 模型评估：对数似然（Log-Likelihood）

**概念解释：** 如何衡量模型的好坏？我们用**似然函数（Likelihood）**：给定训练数据，模型分配给它们的总概率越高，模型越好。

由于概率连乘容易下溢（变成极小的数），我们取对数将乘法变加法：

$$\log \mathcal{L} = \sum_{(x,y) \in \text{data}} \log P(y \mid x)$$

实际使用**负对数似然（Negative Log-Likelihood, NLL）** 作为损失函数——越小越好：

$$\text{NLL} = -\log \mathcal{L}$$

**生活类比：** 类似于考试评分——给模型看它认为"正确答案"的概率有多高。概率高 = 分数低（loss 小）= 模型好。

In [ ]:
# ============================================================
# 计算前 3 个单词的对数似然（Log-Likelihood）
# 对每个 bigram，查概率表 P 得到模型预测的概率
# log(概率) 越接近 0 → 概率越接近 1 → 模型越好
# ============================================================
log_likelihood = 0.0                              # 累计对数似然
for w in word[:3]:                                # 取前 3 个单词
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]                        # 模型预测的 bigram 概率
        logprob = torch.log(prob)                 # 取对数（概率越高，log 越接近 0）
        log_likelihood += logprob                 # 累加
        # N[ix1, ix2]+=1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll = -log_likelihood                             # 负对数似然（Negative Log-Likelihood）
print(f'negative log likelihood: {nll:.4f}')
anll = nll\len(word[:3])                          # 平均负对数似然 = 损失函数
print(f'average negative log likelihood: {anll:.4f}')

In [ ]:
# ============================================================
# 测试罕见 bigram "qg"——在数据集中 q 后面从未出现过 g
# P[q,g] = 0 → log(0) = -inf → NLL 爆炸！
# 这说明未平滑的模型无法处理训练集中没见过的组合
# ============================================================
log_likelihood = 0.0
for w in ["qg"]:                                  # 测试罕见 bigram "qg"
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        # N[ix1, ix2]+=1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll = -log_likelihood
print(f'negative log likelihood: {nll:.4f}')
anll = nll\len(word[:3])
print(f'average negative log likelihood: {anll:.4f}')

### 平滑处理（Smoothing）

**问题：** 如果某个 bigram 在训练数据中从未出现过（比如 `qg`），概率为 0，`log(0) = -inf`，损失直接爆炸！这是因为模型"太绝对"了——只要没见过就判定概率为零。

**解决方案：** 拉普拉斯平滑（Laplace Smoothing / Add-k Smoothing）——给每个计数加一个常数 $k$：

$$P_{ij} = \frac{N_{ij} + k}{\sum_j (N_{ij} + k)}$$

逐项解读：
- $N_{ij}$ = 原始计数（可能为 0）
- $k$ = 平滑常数（通常取 1）
- 分母加了 $k$ 是因为每个位置都加了 $k$，总和也要相应增加
- $k=1$ → 轻微平滑，基本保持原始分布，只消除零概率
- $k \to \infty$ → 概率趋向均匀分布 $1/27$，完全忽略数据

**直觉理解：** 平滑相当于"谦虚"——承认即使训练数据中没出现过的组合，也不能100%排除其可能性。这是概率建模中一个普遍原则：**永远不要给任何事件分配零概率**。

下面实验不同的 $k$ 值对罕见 bigram `"qg"` 损失的影响。

In [ ]:
# ============================================================
# 拉普拉斯平滑（Add-k Smoothing）
# 给每个计数 +1（或更大的 k），消除零概率问题
# 试试不同的 k 值，观察对损失的影响：
#   k=1     → 轻微平滑，基本保持原始分布
#   k=大数  → 概率趋向均匀分布 1/27
# ============================================================
P = (N+1).float()                                 # +1 平滑：消除零概率
# P=(N+500).float()
# P=(N+1000).float()
# P=(N+1000000).float()                           # 加极大平滑值 → 概率趋向均匀分布
# P=torch.ones_like(P)                            # 完全均匀分布（忽略数据统计）
P = P / P.sum(1, keepdim=True)                    # 重新归一化（平滑后要重算概率！）
# P

In [76]:
log_likelihood=0.0
for w in ["qg"]:
    chs=['.']+list(w)+['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]
        logprob=torch.log(prob)
        log_likelihood+=logprob
        # N[ix1, ix2]+=1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll=-log_likelihood
print(f'negative log likelihood: {nll:.4f}')
anll=nll/len(word[:3])
print(f'average negative log likelihood: {anll:.4f}')

log likelihood: -14.4295
negative log likelihood: 14.4295
average negative log likelihood: 4.8098


## 6. 神经网络方法：从查表到学习

**概念解释：** 前面的统计方法直接从数据"数"出概率——简单直接，但无法推广。现在我们换一种方式：用**神经网络学习**这些概率。这个转换是整节课最重要的思想跳跃。

**直觉理解：** 统计方法是"直接数投票"，神经网络是"训练一个裁判来预测投票结果"。在简单的 bigram 场景下，两者结果等价。但当问题变复杂（比如看前 10 个字符预测下一个），你没法用一个 $27^{10}$ 大小的表来存所有情况——统计方法崩溃了，而神经网络依然能通过梯度下降学习。

核心思路（Softmax 回归）：
1. **输入：** 将字符索引转为 **one-hot 向量**（27 维，只有一位是 1）
2. **线性层：** 乘以权重矩阵 $W$，得到 **logits**（未归一化的分数）
3. **Softmax：** 对 logits 取 $\exp$ 再归一化，得到概率分布

$$\text{logits} = x_{\text{one-hot}} \cdot W$$
$$P(y \mid x) = \text{softmax}(\text{logits}) = \frac{e^{\text{logits}_y}}{\sum_j e^{\text{logits}_j}}$$

逐项解读：
- $x_{\text{one-hot}}$ = 输入字符的 one-hot 编码，形状 `(1, 27)`
- $W$ = 可学习的权重矩阵，形状 `(27, 27)`
- $\text{logits}$ = 线性层输出，可以是任意实数（正、负、大、小）
- $e^{\text{logits}}$ = 取指数保证非负
- 除以总和保证归一化（和为 1）
- 整个 Softmax 做的事：**把任意实数变成合法概率分布**

**为什么需要 Softmax？** 神经网络输出的 logits 可以是任意实数，但概率必须满足两个条件：非负、和为1。`exp()` 保证非负，除以总和保证归一化。这和统计方法中的 `count / count.sum()` 本质上是同一件事——只是"计数"从真实的出现次数变成了 `exp(logits)` 这个"伪计数"。

- **One-hot 编码：** 把离散的字符变成数值向量，让矩阵乘法能处理
- **Logits → Softmax：** 将任意实数变成合法概率分布（非负、和为 1）

In [ ]:
# ============================================================
# 构建训练集：将名字拆成 bigram 输入-输出对
# 每个 bigram (ch1→ch2) 变成一个训练样本：x=ch1 的索引, y=ch2 的索引
# 'emma' → 5 个样本：(.→e), (e→m), (m→m), (m→a), (a→.)
# ============================================================
xs, ys = [], []                                   # xs: 输入字符索引，ys: 目标字符索引
for w in word[:1]:                                # 先用第 1 个单词 'emma' 测试
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):            # 取 bigram 对
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(f'{ch1}{ch2}: {ix1} {ix2}')
        xs.append(ix1)                            # 输入：当前字符
        ys.append(ix2)                            # 标签：下一个字符

xs = torch.tensor(xs)                             # 转为 PyTorch 张量
ys = torch.tensor(ys)
xs, ys

In [78]:
import torch.nn.functional as F # 导入 PyTorch 的函数库，常用的函数如 softmax、cross_entropy 等都在这里面


### One-hot 编码（独热编码）

神经网络的输入必须是数值向量，不能是整数索引。**One-hot 编码**把一个整数 $k$ 变成一个长度为 27 的向量：第 $k$ 位是 1，其余全是 0。

**生活类比：** 想象 27 盏灯排成一排，每次只亮一盏——亮哪盏就代表哪个字符。这种编码看似浪费（27 个数只传递 1 个信息），但它的优点是让矩阵乘法 `xenc @ W` 变成了"查表"操作：one-hot 向量乘以 $W$ 等价于直接取 $W$ 的对应行。

In [ ]:
# one-hot 编码示例：索引 0（'.'）→ [1, 0, 0, ..., 0]（27 维向量，只有第 0 位是 1）
# .float() 是因为矩阵乘法需要浮点数（one_hot 默认返回整数）
F.one_hot(xs[:1], num_classes=27).float()

In [ ]:
# 对比：标签 ys[:1]=5（'e'）→ [0, 0, 0, 0, 0, 1, 0, ..., 0]（第 5 位是 1）
F.one_hot(ys[:1], num_classes=27).float()

In [ ]:
# ============================================================
# 将所有 5 个输入索引 one-hot 编码，得到 (5, 27) 矩阵
# 可视化：每行只有一个白色像素（=1），其余为黑色（=0）
# ============================================================
xenc = F.one_hot(xs, num_classes=27).float()      # (5, 27) — 5 个样本，每个 27 维
xenc
xenc.shape                                        # torch.Size([5, 27])
plt.imshow(xenc)                                  # 可视化 one-hot 矩阵

In [ ]:
# 标签也 one-hot 编码用于可视化对比（实际训练中标签不需要 one-hot）
yenc = F.one_hot(ys, num_classes=27).float()
yenc
yenc.shape
plt.imshow(yenc)

### 线性层与权重矩阵 W

神经网络的核心运算：`logits = xenc @ W`。权重矩阵 $W$ 的形状是 `(27, 输出维度)`。输出维度 = 27（因为我们要预测 27 个字符中的哪一个）。先用小矩阵 `(27,2)` 验证形状，再换成正式的 `(27,27)`。

In [ ]:
# 试验：用小权重矩阵验证矩阵乘法的形状
# W=torch.randn((27,1))                          # 输出 1 维（不够用）
W = torch.randn((27,2))                          # 输出 2 维（仅用于理解形状）
xenc @ W                                          # (5,27) @ (27,2) → (5,2)

### 回顾训练数据：xs 与 ys

在进入完整的前向传播之前，先确认我们的训练数据格式。`xs` 是输入字符的索引序列，`ys` 是对应的目标字符索引序列。每一对 `(xs[i], ys[i])` 就是一个 bigram 训练样本。

In [ ]:
xs, ys  # 输入索引 vs 目标索引，一一对应

### Softmax 前向传播与损失计算

下面我们用完整的 27×27 权重矩阵，跑一遍前向传播流程，并逐步计算每个 bigram 的负对数似然损失（NLL）。这有助于理解神经网络是如何"打分"的。

In [ ]:
# 初始化 27×27 权重矩阵（随机正数，因为 torch.rand 返回 [0,1) 均匀分布）
# 注意：这里用 rand 而不是 randn，所以初始 logits 全为正数
g = torch.Generator().manual_seed(2147483647)
w = torch.rand((27,27), generator=g)

In [ ]:
# ============================================================
# Softmax 前向传播完整流程（4 步）：
#   1. one-hot 编码输入
#   2. 矩阵乘法得到 logits（未归一化的分数）
#   3. exp() 把 logits 变成正数（类似"伪计数"）
#   4. 除以行和得到概率分布（这就是 Softmax！）
# ============================================================
xenc = F.one_hot(xs, num_classes=27).float()      # (5,27) one-hot 矩阵
logits = xenc @ w                                 # (5,27) @ (27,27) → (5,27) logits
counts = logits.exp()                             # exp 将 logits 转为正数
probs = counts / counts.sum(1, keepdim=True)      # 归一化为概率（Softmax）

In [ ]:
# 查看中间结果的形状和数值：
# logits: 原始分数（可正可负）
# counts: exp(logits)，全为正数
# probs: 归一化后的概率，每行和为 1
logits, logits.shape, counts, counts.shape, probs, probs.shape

In [ ]:
# ============================================================
# 逐个 bigram 计算负对数似然（NLL），展示完整推理过程
# 对每个样本：模型给正确字符的概率越高 → NLL 越低 → 越好
# ============================================================
nlls = torch.zeros(5)
for i in range(5):                                    # 逐个 bigram 计算损失
    # i-th bigram:
    x = xs[i].item()                                  # 输入字符索引
    y = ys[i].item()                                  # 标签字符索引
    print(f'bigram example {i+1}: ({itos[x]} -> {itos[y]}) (indexes: ({x}, {y}))')
    print('input to the neural net:', x)
    print('output probabilities from the neural net:', probs[i])
    print('label (actual next character):', y)
    p = probs[i, y]                                   # 模型给正确字符分配的概率
    print('probability assigned by the net to the correct character:', p.item())
    logp = torch.log(p)                               # 取对数
    print('log likelihood:', logp.item())
    nll = -logp                                       # 负对数似然
    print('negative loglikelihood:', nll.item())
    nlls[i] = nll
    print('---')
    print('---')


print('average negative log likelihood:', nlls.mean().item())  # 平均 NLL = 损失

## 7. 梯度下降优化（Gradient Descent）

**概念解释：** 现在我们有了损失函数（NLL），如何让模型变好？答案是**梯度下降（Gradient Descent）**——机器学习中最核心的优化算法。

**直觉理解：** 梯度告诉你：如果把某个参数稍微增大一点，loss 会怎么变。梯度为正 → 增大参数会增大 loss → 所以要往负方向走。这就是为什么更新公式是 `W -= lr * grad` 而不是 `+=`。

完整流程（每轮迭代）：
1. **前向传播（Forward Pass）：** 计算预测概率和损失
2. **反向传播（Backpropagation）：** 计算损失对每个权重的梯度 $\frac{\partial L}{\partial W}$
3. **参数更新：** $W \leftarrow W - \eta \cdot \nabla_W L$（$\eta$ 是学习率）

$$W_{\text{new}} = W_{\text{old}} - \eta \cdot \frac{\partial \text{Loss}}{\partial W}$$

逐项解读：
- $W_{\text{old}}$ = 当前权重
- $\eta$ = 学习率（learning rate），控制每步走多远
- $\frac{\partial \text{Loss}}{\partial W}$ = 梯度，指向 loss 增大最快的方向
- 减号：沿梯度**反方向**走 → loss 减小

**生活类比：** 像蒙眼下山——你看不到山的全貌，但能感受脚下的坡度（梯度），每一步沿着最陡的下坡方向走一小步，最终到达山谷（最低损失）。

- 学习率 $\eta$ 太大 → 步子太大，直接跨过谷底跳到对面山坡，来回震荡
- 学习率 $\eta$ 太小 → 步子太小，走一百步还没出发点附近，收敛极慢
- 学习率刚好 → 稳步下降，几十步就到谷底

### 梯度下降实战：从单步到训练循环

下面我们先用 `emma` 的 5 个 bigram 做单步实验，确认前向传播、反向传播、参数更新的流程正确，然后切换到完整数据集进行多轮训练。

In [ ]:
xs, ys  # 确认训练数据：输入索引 vs 目标索引

In [ ]:
# requires_grad=True 告诉 PyTorch：追踪这个张量上的所有运算
# 这样 loss.backward() 时才能计算 dLoss/dW
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27,27), generator=g, requires_grad=True)

In [ ]:
# ============================================================
# 前向传播 + 损失计算（向量化写法，不再逐个循环）
# probs[torch.arange(5), ys] 精妙地从每行取出"正确标签"对应的概率
# 例如：第 0 个样本标签是 5 → 取 probs[0, 5]
# ============================================================
xenc = F.one_hot(xs, num_classes=27).float()
logits = xenc @ W                                 # 前向传播：线性层
counts = logits.exp()                             # Softmax 第一步：取指数
probs = counts / counts.sum(1, keepdim=True)      # Softmax 第二步：归一化


loss = -probs[torch.arange(5), ys].log().mean()   # 负对数似然损失（向量化写法）
loss

In [ ]:
# 梯度下降三步曲：
W.grad = None                   # 1. 清零梯度（PyTorch 默认累加梯度，不清零会错）
loss.backward()                 # 2. 反向传播：自动计算 dLoss/dW（填入 W.grad）
W.data += -0.5 * W.grad        # 3. 参数更新：W = W - lr * grad（lr=0.5）
# 注意用 W.data 而不是 W，避免被 autograd 追踪这次更新

### 扩展到完整数据集 + 训练循环

单步实验验证了流程正确后，现在我们：
1. 用全部 32,000+ 名字（~228K 个 bigram）构建训练集
2. 在 for 循环中重复"前向传播 → 反向传播 → 参数更新" 150 轮
3. 加入 **L2 正则化**（`0.01*(W**2).mean()`）防止过拟合——它惩罚过大的权重，效果等价于统计方法中的平滑

**类比：** 单步优化像"试着走一步看看方向对不对"，训练循环像"沿着这个方向坚定地走 150 步"。

In [ ]:
# ============================================================
# 切换到完整数据集：用所有 32,000+ 名字构建训练集
# 之前只用了 'emma'（5 个样本），现在用全部（~228K 个样本）
# ============================================================
xs, ys = [], []                                       # xs: 输入字符索引，ys: 目标字符索引
# for w in word[:2]:                                  # 旧：只用前几个单词
for w in word:                                        # 新：用全部单词
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):                # 取 bigram 对
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(f'{ch1}{ch2}: {ix1} {ix2}')
        xs.append(ix1)                                # 输入：当前字符
        ys.append(ix2)                                # 标签：下一个字符

xs = torch.tensor(xs)                                 # 转为 PyTorch 张量
ys = torch.tensor(ys)
# xs, ys

In [ ]:
xs.nelement()  # 训练样本总数：228,146 个 bigram（所有名字拆出的字符对）

In [ ]:
# ============================================================
# 完整训练循环：150 轮迭代优化权重 W
# 每轮：前向传播 → 计算损失 → 反向传播 → 更新参数
# 损失 = NLL + L2 正则化（防止权重过大导致过拟合）
# ============================================================
LOSS = []                                                # 记录每轮损失用于绘图
NUM_ITER = 150
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((27,27), generator=g, requires_grad=True)
for i in range(NUM_ITER):
    # --- 前向传播 ---
    xenc = F.one_hot(xs, num_classes=27).float()         # one-hot 编码所有输入
    logits = xenc @ W                                    # 线性层
    counts = logits.exp()                                # Softmax step 1
    probs = counts / counts.sum(1, keepdim=True)         # Softmax step 2

    # loss=-probs[torch.arange(xs.nelement()), ys].log().mean()
    loss = -probs[torch.arange(xs.nelement()), ys].log().mean() + 0.01*(W**2).mean()  # NLL + L2 正则化

    # print(f'loss: {loss.item():.4f}')
    # --- 反向传播 ---
    W.grad = None                                        # 清零梯度
    loss.backward()                                      # 计算梯度
    # --- 参数更新 ---
    W.data += -5 * W.grad                                # 学习率 = 5（较大，因为梯度被 228K 样本平均过）
    LOSS.append(loss.item())


plt.plot(range(NUM_ITER), LOSS)                          # 绘制损失曲线（应单调下降）
# probs[torch.arange(5),ys].log()
loss

### 从训练好的神经网络采样

训练完成后，我们用学到的权重 $W$ 来生成名字。采样过程和之前相同：从 `'.'` 出发，每次用 Softmax 计算下一个字符的概率，按概率抽样，直到抽到 `'.'` 结束。

注意：这里不再查表 `P[ix]`，而是通过 `one-hot @ W → softmax` 计算概率——同样的结果，但方法可以推广到更复杂的网络。

> **深入理解：统计方法与神经网络的等价性**
>
> Karpathy 在视频中特别强调的一点：**这个单层 Softmax 神经网络，在训练充分后，
> 会收敛到和统计计数法几乎相同的结果。** 这不是巧合——数学上可以证明，
> 当 one-hot 输入乘以权重矩阵 $W$ 时，`xenc @ W` 的第 $i$ 行实际上就是
> $W$ 的第 $i$ 行。Softmax 把这一行变成概率分布，而梯度下降的目标是最小化 NLL，
> 这等价于最大化似然——和直接从数据统计频率是同一件事。
>
> 那为什么还要用神经网络？因为当模型变复杂（多层、embedding、注意力机制），
> 统计方法无法扩展，但梯度下降依然适用。bigram 只是一个"玩具级"的例子，
> 让我们在最简单的场景下理解：**损失函数（NLL）+ 梯度下降 = 通用的学习框架**，
> 而计数统计只是这个框架在最简单情况下的特例。
>
> L2 正则化项 `0.01*(W**2).mean()` 的作用等价于统计方法中的平滑（smoothing）：
> 它惩罚过大的权重，防止模型对训练数据过拟合，使概率分布更"平滑"。

In [ ]:
# ============================================================
# 从训练好的神经网络采样生成名字
# 流程和统计方法相同，但概率不是查表 P[ix]，
# 而是通过 one-hot → W → softmax 实时计算
# ============================================================
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    out = []
    ix = 0                                               # 从起始符 '.' 开始
    while True:
        #-----
        # before
        # p=P[ix]                                        # 旧方法：直接查概率表

        #-----
        # after                                          # 新方法：通过神经网络计算概率

        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()  # one-hot 编码当前字符
        logits = xenc @ W                                # 前向传播
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)         # Softmax 得到概率分布



        
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率采样
        
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))